# **[PROJECT 1] RAG · Text2SQL 기반 데이터 조회 시스템**

============================================================
[프로젝트] 2026년 2차 서울시 청년안심주택(공공임대) 청약 도우미
------------------------------------------------------------
| 일차 | 역할 | 데이터 |
|------|------|--------|
| **day2** | PDF 공고문 RAG | 모집공고문 PDF (비정형) |
| **day3** | 테이블 Text2SQL | CSV 4종 (정형) |

## 과제 진행 단계
1. **프로젝트 주제 확인** — day2 RAG 주제 유지 + PDF와 연계되는 테이블 정의
2. **테이블 데이터 수집** — CSV 4종 (단지정보, 자격요건, 문의처, 지역정보)
3. **데이터베이스 적재** — Supabase(PostgreSQL) + 로컬 SQLite
4. **Text2SQL 테스트** — 자연어 → SQL → 결과 검증
5. **(시간 여유 시) PDF + DB 활용 설계** — 질문 유형별 라우팅

---

# 1. 프로젝트 주제 확인

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"✓ .env 로드: {dotenv_path}")
else:
    load_dotenv()

print("\n=== 환경 변수 확인 ===")
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key 설정됨")
else:
    print("✗ OpenAI API Key가 없습니다.")

if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL 설정됨")
else:
    print("✗ Supabase DB URL 미설정 (Part 2에서 필요)")

In [ ]:
# =========================================================
# 1-1. 서비스 주제 & 테이블 데이터 정의
# =========================================================
PROJECT_TOPIC = "2026년 2차 서울시 청년안심주택(공공임대) 청약 도우미"

# CSV 파일 ↔ DB 테이블 ↔ 한글명 매핑
TABLES = {
    "apartment": {
        "csv": "apartment.csv",
        "label": "단지정보",
        "description": "단지별 공급유형·임대보증금·월임대료·공급호수",
    },
    "preferences": {
        "csv": "preferences.csv",
        "label": "자격요건",
        "description": "신청자격·순위·소득·자산·가점·임대료구분 매핑 기준",
    },
    "service_center": {
        "csv": "service_center.csv",
        "label": "문의처",
        "description": "공공/민간임대 상담센터·콜센터 연락처",
    },
    "information": {
        "csv": "information.csv",
        "label": "지역정보",
        "description": "시군구별 당해(지역우선) 거주조건·지자체 주거지원·문의창구",
    },
}

print(f"프로젝트: {PROJECT_TOPIC}\n")
print("테이블 데이터:")
for name, info in TABLES.items():
    print(f"  • {info['label']} ({name}) ← {info['csv']}")
    print(f"    └ {info['description']}")

In [ ]:
# =========================================================
# 1-2. Use Case 예시 질문 (난이도별)
# =========================================================
USE_CASE_QUESTIONS = [
    {
        "level": "★",
        "type": "단순 조회",
        "question": "청년안심주택 민간임대 계약 관련해서 문의하려면 어디로 전화해야 하나요?",
        "tables": ["문의처"],
        "note": "단일 행 조회",
    },
    {
        "level": "★",
        "type": "단순 조회",
        "question": "동작구에 사는 청년이 받을 수 있는 지자체 주거 지원과 문의창구는 어디인가요?",
        "tables": ["지역정보"],
        "note": "information.csv 단일 행 조회",
    },
    {
        "level": "★",
        "type": "단순 조회",
        "question": "에이트플레이스 39A 타입(신혼Ⅰ) 임대보증금은 얼마인가요?",
        "tables": ["단지정보"],
        "note": "단일 행 조회",
    },
    {
        "level": "★★",
        "type": "조건 조회",
        "question": "청년 1순위 자격요건에 해당하는 경우는 어떤 경우들이 있나요?",
        "tables": ["자격요건"],
        "note": "동일 테이블 내 여러 행 취합",
    },
    {
        "level": "★★",
        "type": "계산/비교",
        "question": "스타팰리스 이수 18.08형 청년 대상 A유형과 B유형의 월임대료 차이는 얼마인가요?",
        "tables": ["단지정보"],
        "note": "단일 테이블 내 수치 비교·계산",
    },
    {
        "level": "★★★",
        "type": "복합 추론",
        "question": "1인가구 월소득 250만원인 청년이 특별한 감면 요건 없이 스타팰리스 이수에 지원하면 몇 순위이고, 어떤 임대료 유형(A/B)을 적용받나요?",
        "tables": ["자격요건(소득기준)", "단지정보(임대료구분)"],
        "note": "소득 구간→순위 판정→임대료 유형 매핑, 2개 테이블",
    },
    {
        "level": "★★★",
        "type": "복합 추론",
        "question": "신혼부부Ⅱ 4인가구이고 월소득 700만원인 경우, 에이트플레이스 39D 타입 신청 시 적용되는 임대료구분은 무엇이며, 계약 관련 법률상담이 필요할 경우 연락처는 어디인가요?",
        "tables": ["자격요건(소득기준)", "단지정보", "문의처"],
        "note": "3개 테이블, 가장 복잡한 케이스",
    },
    {
        "level": "★★",
        "type": "부재 확인 (edge case)",
        "question": "청년안심주택 재계약 시 위약금 규정은 어떻게 되나요?",
        "tables": [],
        "note": "CSV에 없음 → '정보 없음' 응답 (환각 방지 테스트)",
        "expect_no_data": True,
    },
]

print(f"총 {len(USE_CASE_QUESTIONS)}개 Use Case 질문\n")
for i, uc in enumerate(USE_CASE_QUESTIONS, 1):
    tables = ", ".join(uc["tables"]) if uc["tables"] else "(없음)"
    print(f"{i}. [{uc['level']} {uc['type']}] {uc['question']}")
    print(f"   테이블: {tables} | {uc['note']}\n")

---

# 2. 테이블 데이터 수집

day3 폴더의 CSV 4종을 로드합니다. PDF 공고문(day2)과 **유기적으로 연결**되는 정형 데이터입니다.

| 한글명 | CSV | 예시 질문 |
|--------|-----|-----------|
| 단지정보 | `apartment.csv` | "에이트플레이스 39A 보증금은?" |
| 자격요건 | `preferences.csv` | "청년 1순위 해당 경우는?" |
| 문의처 | `service_center.csv` | "민간임대 계약 문의 전화번호?" |
| 지역정보 | `information.csv` | "동작구 청년 주거지원·문의창구는?" |

In [ ]:
import sqlite3
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, NOTEBOOK_DIR / "project" / "day3"]:
    if (candidate / "apartment.csv").exists():
        DATA_DIR = candidate
        break
else:
    DATA_DIR = NOTEBOOK_DIR

APARTMENT_RENAME = {
    "구분": "category",
    "단지명": "complex_name",
    "주소": "address",
    "공급유형(㎡)": "supply_area_type",
    "평면유형(㎡)": "floor_plan_area",
    "신청자격": "eligibility",
    "공급호수(실)": "units",
    "임대료구분": "rent_grade",
    "임대보증금_계(천원)": "deposit_total_k",
    "계약금_20%(천원)": "contract_deposit_k",
    "잔금_80%(천원)": "balance_k",
    "월임대료(원)": "monthly_rent",
}

PREFERENCES_RENAME = {
    "신청자격": "eligibility",
    "구분유형": "category_type",
    "코드": "code",
    "세부항목": "item",
    "기준값": "criteria_value",
    "비고": "note",
}

SERVICE_CENTER_RENAME = {
    "구분": "category",
    "기관/부서명": "organization",
    "담당업무": "service",
    "전화번호": "phone",
    "비고": "note",
}

INFORMATION_RENAME = {
    "시/도": "sido",
    "시/군/구": "sigungu",
    "청년 청약 당해(지역우선) 거주조건 및 특징": "local_priority",
    "지자체 특화 주거 지원 (월세·보증금 대출이자·이사비 등)": "local_support",
    "세부 신청 자격 (연령·소득·무주택 기준)": "local_eligibility",
    "신청 및 문의 창구 (온라인 포털 / 담당부서 / 연락처)": "contact",
}

rename_map = {
    "apartment": APARTMENT_RENAME,
    "preferences": PREFERENCES_RENAME,
    "service_center": SERVICE_CENTER_RENAME,
    "information": INFORMATION_RENAME,
}

dataframes = {}
for table, info in TABLES.items():
    path = DATA_DIR / info["csv"]
    if not path.exists():
        raise FileNotFoundError(f"CSV 없음: {path}")
    df = pd.read_csv(path, encoding="utf-8-sig").rename(columns=rename_map[table])
    dataframes[table] = df
    print(f"✓ [{info['label']}] {table}: {len(df)}행")
    print(f"  컬럼: {list(df.columns)}")
    print(df.head(2).to_string(index=False))
    print()

---

# 3. 데이터베이스 적재

## 3-1. 로컬 SQLite (개발·테스트용)

In [ ]:
DB_DIR = DATA_DIR / "database"
DB_DIR.mkdir(exist_ok=True)
db_path = DB_DIR / "youth_housing.db"

conn = sqlite3.connect(db_path)
for table, df in dataframes.items():
    df.to_sql(table, conn, if_exists="replace", index=False)
conn.close()

print(f"✓ SQLite DB 생성: {db_path}")

In [ ]:
from langchain_community.utilities import SQLDatabase

db_url = f"sqlite:///{db_path}"
db = SQLDatabase.from_uri(db_url)

print("=== SQLite 연결 ===")
print(f"테이블: {db.get_usable_table_names()}")

## 3-2. SQL 조회 테스트 (직접 작성)

In [ ]:
# ★ 단순 조회 — 문의처
q1 = """
SELECT organization, service, phone
FROM service_center
WHERE category = '민간임대'
  AND service LIKE '%계약%';
"""
print("[★] 민간임대 계약 문의처")
print(db.run(q1))

# ★ 단순 조회 — 단지정보
q2 = """
SELECT complex_name, supply_area_type, eligibility, rent_grade, deposit_total_k
FROM apartment
WHERE complex_name LIKE '%에이트플레이스%'
  AND supply_area_type = '39A'
  AND eligibility = '신혼Ⅰ';
"""
print("\n[★] 에이트플레이스 39A(신혼Ⅰ) 보증금")
print(db.run(q2))

# ★★ 조건 조회 — 자격요건
q3 = """
SELECT code, item, criteria_value, note
FROM preferences
WHERE eligibility = '청년'
  AND category_type = '자격요건'
  AND code = '1순위';
"""
print("\n[★★] 청년 1순위 자격요건")
print(db.run(q3))

# ★ 단순 조회 — 지역정보 (information.csv)
q_info = """
SELECT sido, sigungu, local_support, contact
FROM information
WHERE sigungu LIKE '%동작구%';
"""
print("\n[★] 동작구 지자체 주거지원·문의창구")
print(db.run(q_info))

In [ ]:
# ★★ 계산/비교 — A vs B 월임대료 차이
q4 = """
SELECT rent_grade, monthly_rent,
       MAX(monthly_rent) OVER () - MIN(monthly_rent) OVER () AS diff
FROM apartment
WHERE complex_name LIKE '%스타팰리스 이수%'
  AND supply_area_type LIKE '18.08%'
  AND eligibility = '청년'
  AND rent_grade IN ('A', 'B');
"""
print("[★★] 스타팰리스 이수 18.08형 A/B 월임대료")
print(db.run(q4))

In [ ]:
# ★★★ 복합 추론 — 소득→순위→임대료구분 (힌트 쿼리)
# 250만원(2,500,000) < 청년 3순위 1인가구 100% 기준(4,576,036) → 3순위
# 청년 3순위 → 임대료구분 B (preferences: 임대료구분 B = 2,3순위)

income_check = """
SELECT eligibility, category_type, code, item, criteria_value
FROM preferences
WHERE eligibility = '청년'
  AND category_type IN ('소득기준', '임대료구분', '자격요건')
  AND (item LIKE '%1인가구%' OR code IN ('2순위', '3순위') OR item LIKE '%임대료%');
"""
print("[★★★] 청년 소득·순위·임대료구분 기준 (복합 추론 참고)")
print(db.run(income_check))

apt_check = """
SELECT complex_name, supply_area_type, eligibility, rent_grade, monthly_rent
FROM apartment
WHERE complex_name LIKE '%스타팰리스 이수%'
  AND supply_area_type LIKE '18.08%'
  AND eligibility = '청년';
"""
print("\n[★★★] 스타팰리스 이수 18.08형 청년 임대료 유형")
print(db.run(apt_check))

## 3-3. Supabase PostgreSQL 적재

**Supabase 대시보드에서 CSV 업로드**
1. [Supabase](https://supabase.com/) → Table Editor → Import data from CSV
2. `apartment`, `preferences`, `service_center`, `information` 순서로 업로드
3. 컬럼명은 SQLite와 동일한 **영문명** 사용 권장

In [ ]:
supabase_db_url = os.getenv("SUPABASE_DB_URL")

if supabase_db_url:
    pg_db = SQLDatabase.from_uri(supabase_db_url)
    print("✓ Supabase PostgreSQL 연결 성공")
    print(f"테이블: {pg_db.get_usable_table_names()}")
else:
    pg_db = None
    print("Supabase 미설정 — SQLite로 Part 4까지 진행 가능")

---

# 4. Text2SQL 테스트

## 4-1. Text2SQL 함수 구현

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
import time

llm = init_chat_model("gpt-4o-mini")


def invoke_llm(messages, max_retries=6, base_delay=3.0):
    """RateLimitError(429) 시 지수 백오프로 재시도"""
    last_err = None
    for attempt in range(max_retries):
        try:
            return llm.invoke(messages)
        except Exception as e:
            last_err = e
            err = f"{type(e).__name__} {e}".lower()
            if "ratelimit" not in err and "rate_limit" not in err and "429" not in err:
                raise
            wait = base_delay * (2 ** attempt)
            print(f"    RateLimit — {wait:.0f}초 대기 후 재시도 ({attempt + 1}/{max_retries})")
            time.sleep(wait)
    raise last_err

REASONING_HINT = """
<domain reasoning hints>
- 청년 1순위: 생계·의료·주거급여 수급, 보호대상 한부모, 차상위계층 (자격요건 테이블)
- 청년 2순위: 본인+부모 소득 100% 이하 + 자산기준
- 청년 3순위: 1·2순위 미해당 + 본인 소득 100% 이하(1인가구)
- 청년 임대료구분: 1순위→A(시중시세 30%), 2·3순위→B(시중시세 50%)
- 신혼부부Ⅰ: 소득 50%→A, 70%→B, 90%→B
- 신혼부부Ⅱ: 소득 80%→C, 130%→D
- 금액: deposit_total_k=천원, monthly_rent=원
- 소득기준 비교 시 criteria_value에서 숫자만 추출해 비교
</domain reasoning hints>
"""

def text_to_sql(question: str, db: SQLDatabase) -> str:
    system_prompt = f"""
당신은 청년안심주택 청약 데이터 SQL 전문가입니다.
사용자 질문을 SQLite SELECT 쿼리로 변환하세요.

<database schema>
{db.table_info}
</database schema>

<table guide>
- apartment (단지정보): complex_name, supply_area_type, eligibility, rent_grade, deposit_total_k, monthly_rent
- preferences (자격요건): eligibility, category_type, code, item, criteria_value, note
- service_center (문의처): category, organization, service, phone, note
- information (지역정보): sido, sigungu, local_priority, local_support, local_eligibility, contact
</table guide>

{REASONING_HINT}

<rules>
- SQLite 문법, SELECT만 허용
- SQL 코드만 반환 (설명·코드블록 없음), 세미콜론(;)으로 종료
- JOIN, 서브쿼리, GROUP BY, CASE WHEN 사용 가능
- 복합 추론 질문은 preferences에서 기준 조회 후 apartment/service_center와 연결
</rules>
"""

    response = invoke_llm([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ])
    sql = response.content.strip()
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()
    return sql

print("✓ Text2SQL 함수 준비 완료")

## 4-2. Text2SQL + 자연어 답변 시스템

In [ ]:
def _is_empty_result(result: str) -> bool:
    if not result or not str(result).strip():
        return True
    normalized = str(result).strip()
    return normalized in ("[]", "()", "[()]", "None")


def query_database(question: str, db: SQLDatabase, expect_no_data: bool = False) -> str:
    """자연어 → SQL → 실행 → 자연어 답변 (환각 방지 포함)"""
    print("[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    print("[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    결과: {result}\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    if _is_empty_result(result):
        print("[3] 조회 결과 없음 → 환각 방지 응답")
        if expect_no_data:
            return (
                "제공된 테이블 데이터(CSV)에는 해당 정보가 없습니다. "
                "재계약 위약금 등 계약 세부 규정은 day2 PDF 공고문 RAG에서 확인하시거나 "
                "SH공사 콜센터(1600-3456)에 문의해 주세요."
            )
        return (
            "제공된 데이터베이스에서 해당 정보를 찾을 수 없습니다. "
            "질문을 다시 확인하시거나 day2 PDF 공고문에서 추가 정보를 검색해 보세요."
        )

    print("[3] 답변 생성 중...")
    system_prompt = """
당신은 2026년 2차 서울시 청년안심주택(공공임대) 청약 안내 전문가입니다.
SQL 쿼리 결과만을 근거로 답변하세요. 결과에 없는 내용은 추측하지 마세요.
금액 단위(원/천원)를 명확히 구분하세요.
복합 추론 질문은 순위 판정 → 임대료구분 매핑 과정을 단계별로 설명하세요.
"""

    response = invoke_llm([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
<question>{question}</question>
<sql>{sql}</sql>
<result>{result}</result>
위 SQL 결과만 근거로 질문에 답변하세요.
"""),
    ])
    return response.content

print("✓ query_database 준비 완료")

## 4-3. Use Case 질문별 Text2SQL 테스트

In [ ]:
from IPython.display import Markdown, display
import time

for i, uc in enumerate(USE_CASE_QUESTIONS, 1):
    print(f"\n{'=' * 80}")
    print(f"[{i}] {uc['level']} {uc['type']}")
    print(f"질문: {uc['question']}")
    print(f"필요 테이블: {', '.join(uc['tables']) if uc['tables'] else '(없음)'}")
    print("=" * 80 + "\n")

    answer = query_database(
        uc["question"],
        db,
        expect_no_data=uc.get("expect_no_data", False),
    )
    print("답변:")
    display(Markdown(answer))

    if i < len(USE_CASE_QUESTIONS):
        time.sleep(2)

---

# 5. (시간 여유 시) PDF + DB 활용 설계

day2 RAG(PDF)와 day3 Text2SQL(DB)을 **어떤 질문에 쓸지** 라우팅합니다.

In [ ]:
# =========================================================
# 5-1. 질문 유형별 라우팅 규칙
# =========================================================
ROUTING_RULES = [
    {
        "route": "DB (Text2SQL)",
        "examples": [
            "에이트플레이스 39A 보증금/월세",
            "단지별 공급호수·임대료 유형",
            "청년 1순위 자격요건 목록",
            "상담센터 전화번호",
        ],
        "reason": "정형 테이블에 정확한 수치·연락처가 있음",
    },
    {
        "route": "PDF (RAG)",
        "examples": [
            "청약 접수 일정",
            "제출 서류 목록",
            "재계약·위약금 규정",
            "셰어형 2인 1팀 신청 방법",
        ],
        "reason": "공고문 서술형·절차 정보",
    },
    {
        "route": "DB + PDF (복합)",
        "examples": [
            "250만원 소득 청년 순위 + 스타팰리스 지원 방법",
            "신혼부부Ⅱ 소득 기준 + 계약 시 유의사항",
        ],
        "reason": "DB로 순위·금액 판정 → PDF로 절차·서류 안내",
    },
]

for rule in ROUTING_RULES:
    print(f"\n▶ {rule['route']}")
    print(f"  이유: {rule['reason']}")
    for ex in rule["examples"]:
        print(f"  • {ex}")

In [ ]:
def route_question(question: str) -> str:
    """키워드 기반 간단 라우터 (발표용 프로토타입)"""
    db_keywords = ["보증금", "월세", "월임대료", "공급호수", "전화", "연락처", "순위", "소득", "가점", "임대료구분", "타입", "지자체", "당해", "문의창구"]
    pdf_keywords = ["일정", "접수", "서류", "제출", "위약금", "재계약", "셰어", "방법", "유의사항", "반려동물"]

    db_score = sum(1 for k in db_keywords if k in question)
    pdf_score = sum(1 for k in pdf_keywords if k in question)

    if db_score > 0 and pdf_score > 0:
        return "DB + PDF"
    if db_score >= pdf_score and db_score > 0:
        return "DB"
    if pdf_score > 0:
        return "PDF"
    return "DB + PDF"  # 불명확 시 복합


test_routes = [uc["question"] for uc in USE_CASE_QUESTIONS]
test_routes.append("청약 접수는 언제부터 언제까지인가요?")

print("질문 라우팅 테스트:\n")
for q in test_routes:
    print(f"  [{route_question(q):8s}] {q[:50]}{'...' if len(q) > 50 else ''}")

## 프로젝트 점검 체크리스트

- [ ] day2와 동일 주제 유지 + 테이블 4종 정의 완료
- [ ] CSV 수집 및 데이터 탐색 완료
- [ ] SQLite / Supabase DB 적재 완료
- [ ] 직접 SQL 쿼리 테스트 (★~★★★) 완료
- [ ] Text2SQL 함수 구현 및 Use Case 테스트 완료 (information 포함)
- [ ] edge case(위약금) → "정보 없음" 환각 방지 확인
- [ ] PDF vs DB 라우팅 설계 완료

---

## 발표 포인트

| 난이도 | 질문 유형 | 강조 메시지 |
|--------|-----------|-------------|
| ★ | 단순 조회 | DB가 PDF보다 **정확한 수치**에 강함 |
| ★★ | 조건·계산 | SQL 집계/비교로 **즉시 답변** |
| ★★★ | 복합 추론 | 소득→순위→임대료 **다중 테이블 추론** |
| ★★ edge | 부재 확인 | CSV에 없으면 **"모른다"** — 환각 방지 |